# Technology Continuity Storytelling & Persona Engine
### محرك سرد تفاعلي + تشخيص الجاهزية + تصنيف الشخصيات

هذا الـNotebook يبني تجربة مؤتمر لا تبدو كاستبيان. الزائر يمر عبر **قصة قصيرة من مواقف تقنية محتملة**؛ بينما يقوم المحرك في الخلفية بحساب درجات مخفية، اكتشاف مواضع الألم، تحديد الـPersona، وتقدير الاحتياج التجاري.

**ما يراه الزائر:** Readiness Score + Persona + Priority Area + Recommendation.  
**ما يراه الفريق فقط:** Eligibility + Service Need + Commercial Opportunity + Lead Class.


## 1) تثبيت المكتبات
تشغيل هذه الخلية في Google Colab يثبت المتطلبات الأساسية.

In [ ]:
!pip -q install pandas plotly gspread google-auth streamlit

## 2) الاستيراد والإعداد

In [ ]:
import json, uuid, math, os
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


## 3) هوية القصة
الفكرة أن نبدأ بمشهد وليس بسؤال مباشر عن الألم.

In [ ]:
STORY = {
    "en": {
        "title": "What if tomorrow changed everything?",
        "opening": "Your systems are running. Your provider is responsive. Everything feels under control — until one dependency suddenly is not.",
        "promise": "Walk through a few real-world moments. We will reveal how ready your organization is to stay in control.",
        "chapters": [
            "Your Role", "The First Disruption", "Control of Critical Assets",
            "The Exit Moment", "Proof of Readiness", "Your Continuity Profile"
        ],
        "analyzing": "Connecting the signals behind your answers...",
    },
    "ar": {
        "title": "ماذا لو تغيّر كل شيء غدًا؟",
        "opening": "أنظمتك تعمل، المورد متجاوب، وكل شيء يبدو تحت السيطرة — حتى تتوقف فجأة حلقة واحدة تعتمد عليها.",
        "promise": "مرّ معنا بعدة مواقف واقعية قصيرة، وسنكشف لك مدى جاهزية جهتك للبقاء مسيطرة وقت التعطل أو الخلاف.",
        "chapters": [
            "دورك", "لحظة التعطل", "السيطرة على الأصول",
            "لحظة الخروج", "إثبات الجاهزية", "ملف الاستمرارية"
        ],
        "analyzing": "نربط الإشارات المخفية خلف إجاباتك...",
    }
}


## 4) الأسئلة السيناريوهية والأوزان المخفية
كل اختيار يحمل درجات مخفية. الزائر لا يرى هذه الأوزان.

الأبعاد الأساسية: **Asset Control, Continuity, Exit Readiness, Contract Clarity, Governance, Business Criticality, Provider Assurance**.

In [ ]:
QUESTIONS = [
    {
        "id":"representation", "chapter":0, "paths":["all"],
        "en":"Which role are you stepping into today?",
        "ar":"أي دور تمثله اليوم؟",
        "context_en":"Every continuity story starts with knowing where you sit in the ecosystem.",
        "context_ar":"كل قصة استمرارية تبدأ بفهم موقعك في المنظومة.",
        "options":[
            {"id":"client","en":"I represent an organization that uses or procures technology","ar":"أمثل جهة تستخدم أو تشتري حلولًا تقنية","route":"client","eligibility":30,"fit":20},
            {"id":"provider","en":"We build or provide technology solutions","ar":"نطور أو نقدم حلولًا تقنية","route":"provider","eligibility":25,"fit":18},
            {"id":"hybrid","en":"We both use and provide technology solutions","ar":"نستخدم ونقدم حلولًا تقنية في الوقت نفسه","route":"hybrid","eligibility":30,"fit":20},
            {"id":"advisor","en":"I advise organizations on technology, contracts or risk","ar":"أقدم استشارات للجهات في التقنية أو العقود أو المخاطر","route":"advisor","eligibility":18,"fit":12},
            {"id":"explorer","en":"I am exploring the topic for myself","ar":"أستكشف الموضوع لنفسي فقط","route":"explorer","eligibility":0,"fit":0},
        ]
    },
    {
        "id":"influence", "chapter":0, "paths":["client","provider","hybrid","advisor"],
        "en":"When a critical technology decision is made, where are you usually in the room?",
        "ar":"عندما يُتخذ قرار تقني حرج، أين يكون دورك عادة؟",
        "context_en":"This tells us how close you are to the moment where risk becomes a decision.",
        "context_ar":"هذا يكشف مدى قربك من اللحظة التي يتحول فيها الخطر إلى قرار.",
        "options":[
            {"id":"approve","en":"I approve or own the decision","ar":"أعتمد القرار أو أملكه","influence":25,"eligibility":15},
            {"id":"evaluate","en":"I evaluate and recommend","ar":"أقيّم وأوصي","influence":20,"eligibility":12},
            {"id":"manage","en":"I manage or implement it","ar":"أدير أو أنفذ القرار","influence":15,"eligibility":10},
            {"id":"use","en":"I mainly use the systems","ar":"أستخدم الأنظمة بشكل أساسي","influence":6,"eligibility":5},
            {"id":"none","en":"I am not directly involved","ar":"لست مشاركًا مباشرة","influence":0,"eligibility":0},
        ]
    },
    {
        "id":"org_scale", "chapter":0, "paths":["client","provider","hybrid","advisor"],
        "en":"Which environment sounds closest to the organization you represent?",
        "ar":"أي بيئة أقرب للجهة التي تمثلها؟",
        "context_en":"The same disruption means something very different at different scales.",
        "context_ar":"نفس التعطل قد يعني أثرًا مختلفًا تمامًا باختلاف حجم الجهة.",
        "options":[
            {"id":"enterprise","en":"Large enterprise or government entity","ar":"منشأة كبيرة أو جهة حكومية","fit":20,"eligibility":15},
            {"id":"mid","en":"Medium-sized organization","ar":"منشأة متوسطة","fit":15,"eligibility":10},
            {"id":"small","en":"Small organization or startup","ar":"منشأة صغيرة أو شركة ناشئة","fit":8,"eligibility":5},
            {"id":"individual","en":"Individual / no organization represented","ar":"فرد / لا أمثل جهة","fit":0,"eligibility":0},
        ]
    },
    {
        "id":"criticality", "chapter":1, "paths":["client","hybrid","advisor"],
        "en":"It is 9:00 AM. A critical digital service goes offline. When does the disruption become a serious business issue?",
        "ar":"الساعة 9 صباحًا وتوقفت خدمة رقمية حرجة. متى يصبح التوقف مشكلة فعلية للأعمال؟",
        "context_en":"The clock is now running.",
        "context_ar":"من هذه اللحظة يبدأ عدّاد الأثر.",
        "options":[
            {"id":"minutes","en":"Within minutes","ar":"خلال دقائق","business_criticality":100,"need":22},
            {"id":"hours","en":"Within a few hours","ar":"خلال ساعات قليلة","business_criticality":85,"need":18},
            {"id":"day","en":"By the end of the day","ar":"بنهاية اليوم","business_criticality":65,"need":13},
            {"id":"days","en":"After several days","ar":"بعد عدة أيام","business_criticality":35,"need":6},
            {"id":"minimal","en":"The impact would be limited","ar":"التأثير سيكون محدودًا","business_criticality":10,"need":1},
        ]
    },
    {
        "id":"provider_disruption", "chapter":1, "paths":["client","hybrid","advisor"],
        "en":"The provider is now unreachable. What happens first?",
        "ar":"الآن تعذر الوصول إلى المورد. ماذا يحدث أولًا؟",
        "context_en":"No warning. No time to renegotiate. Only what is already under your control matters.",
        "context_ar":"لا يوجد إنذار مسبق ولا وقت لإعادة التفاوض. الآن لا يفيد إلا ما هو أصلًا تحت سيطرتكم.",
        "options":[
            {"id":"internal","en":"Our team continues using assets and procedures already under our control","ar":"يستمر فريقنا باستخدام الأصول والإجراءات الموجودة تحت سيطرتنا","continuity":95,"asset_control":90,"need":2},
            {"id":"arrangement","en":"We activate a predefined continuity arrangement","ar":"نفعّل ترتيب استمرارية محدد مسبقًا","continuity":85,"asset_control":80,"need":5},
            {"id":"alternate","en":"We coordinate with another provider to restore operations","ar":"نتواصل مع مزود بديل لاستعادة التشغيل","continuity":65,"asset_control":60,"need":10},
            {"id":"wait","en":"We need the existing provider before we can fully recover","ar":"نحتاج المورد الحالي قبل أن نستعيد التشغيل بالكامل","continuity":30,"asset_control":35,"need":20},
            {"id":"unknown","en":"I am not sure what would happen","ar":"لست متأكدًا مما سيحدث","continuity":20,"asset_control":25,"need":22},
        ]
    },
    {
        "id":"handover", "chapter":2, "paths":["client","hybrid","advisor"],
        "en":"A new technical team must take over next week. What could you hand them today?",
        "ar":"يجب أن يستلم فريق تقني جديد النظام الأسبوع القادم. ماذا تستطيعون تسليمه اليوم؟",
        "context_en":"A smooth handover is often the clearest proof of who truly controls the technology assets.",
        "context_ar":"سهولة التسليم غالبًا هي أوضح دليل على من يسيطر فعليًا على الأصول التقنية.",
        "options":[
            {"id":"complete","en":"A complete package: code, data, documentation, access and deployment details","ar":"حزمة مكتملة: الكود والبيانات والتوثيق والصلاحيات وتفاصيل النشر","asset_control":100,"exit_readiness":95,"need":1},
            {"id":"mostly","en":"Mostly complete, with a few provider dependencies","ar":"مكتملة إلى حد كبير مع بعض الاعتماد على المورد","asset_control":75,"exit_readiness":75,"need":7},
            {"id":"partial","en":"Partial; significant coordination is still required","ar":"جزئية؛ وما زلنا نحتاج تنسيقًا كبيرًا","asset_control":50,"exit_readiness":45,"need":14},
            {"id":"obtain","en":"Key assets must first be obtained from the current provider","ar":"نحتاج أولًا الحصول على أصول أساسية من المورد الحالي","asset_control":25,"exit_readiness":25,"need":21},
            {"id":"unknown","en":"I am not sure what is available","ar":"لست متأكدًا مما هو متوفر","asset_control":20,"exit_readiness":20,"need":22},
        ]
    },
    {
        "id":"exit", "chapter":3, "paths":["client","hybrid","advisor"],
        "en":"The relationship ends tomorrow. Which transition feels closest to reality?",
        "ar":"تنتهي العلاقة مع المورد غدًا. أي سيناريو انتقال هو الأقرب للواقع؟",
        "context_en":"Most organizations discover their contract gaps at the worst possible moment: during exit.",
        "context_ar":"كثير من الجهات لا تكتشف فجوات العقود إلا في أصعب لحظة: عند الخروج.",
        "options":[
            {"id":"predefined","en":"Transition steps, responsibilities and access rights are predefined","ar":"خطوات الانتقال والمسؤوليات وحقوق الوصول محددة مسبقًا","exit_readiness":100,"contract_clarity":95,"need":1},
            {"id":"general","en":"The agreement provides general transition guidance","ar":"الاتفاقية توفر توجيهًا عامًا للانتقال","exit_readiness":70,"contract_clarity":70,"need":7},
            {"id":"negotiate","en":"The parties would agree on transition arrangements when needed","ar":"سيتم الاتفاق على ترتيبات الانتقال عند الحاجة","exit_readiness":45,"contract_clarity":45,"need":14},
            {"id":"provider","en":"The transition depends heavily on the provider","ar":"يعتمد الانتقال بدرجة كبيرة على المورد","exit_readiness":25,"contract_clarity":30,"need":21},
            {"id":"unknown","en":"I am not sure","ar":"لست متأكدًا","exit_readiness":20,"contract_clarity":20,"need":22},
        ]
    },
    {
        "id":"assurance", "chapter":2, "paths":["provider","hybrid"],
        "en":"A major client asks: ‘If your company cannot support us tomorrow, how do we stay protected?’ What can you demonstrate?",
        "ar":"يسألك عميل كبير: «إذا تعذر على شركتكم دعمنا غدًا، كيف نبقى محميين؟» ماذا تستطيعون إثباته؟",
        "context_en":"Enterprise trust is strongest when continuity does not depend on promises alone.",
        "context_ar":"ثقة عملاء المؤسسات تكون أقوى عندما لا تعتمد الاستمرارية على الوعود وحدها.",
        "options":[
            {"id":"independent","en":"Independent safeguards, documented handover and controlled asset access","ar":"ضمانات مستقلة وتسليم موثق ووصول منضبط للأصول","continuity":95,"asset_control":95,"provider_assurance":100,"need":2},
            {"id":"internal","en":"Strong internal processes and documented commitments","ar":"إجراءات داخلية قوية والتزامات موثقة","continuity":75,"asset_control":75,"provider_assurance":75,"need":7},
            {"id":"contract","en":"Mainly contractual commitments","ar":"نعتمد بشكل أساسي على الالتزامات التعاقدية","continuity":55,"asset_control":55,"provider_assurance":50,"need":13},
            {"id":"ad_hoc","en":"We would arrange the handover if the situation occurs","ar":"سنرتب التسليم إذا حدث الموقف","continuity":35,"asset_control":35,"provider_assurance":30,"need":19},
            {"id":"unsure","en":"I am not sure how we would demonstrate this","ar":"لست متأكدًا كيف سنثبت ذلك","continuity":25,"asset_control":25,"provider_assurance":20,"need":21},
        ]
    },
    {
        "id":"client_request", "chapter":3, "paths":["provider","hybrid"],
        "en":"A client asks for proof about code, data, backups and continuity. What usually follows?",
        "ar":"طلب عميل إثباتات عن الكود والبيانات والنسخ الاحتياطية والاستمرارية. ماذا يحدث عادة؟",
        "context_en":"This is where trust either accelerates the deal — or slows it down.",
        "context_ar":"هنا إمّا أن تسرّع الثقة الصفقة، أو تبدأ الاحتكاكات والتأخير.",
        "options":[
            {"id":"standard","en":"We have a standard assurance package ready to share","ar":"لدينا حزمة ضمان قياسية وجاهزة للمشاركة","provider_assurance":100,"contract_clarity":90,"need":2},
            {"id":"custom","en":"We prepare evidence for each client","ar":"نجهز الإثباتات حسب كل عميل","provider_assurance":70,"contract_clarity":70,"need":8},
            {"id":"legal","en":"It becomes a contract/legal negotiation","ar":"يتحول الأمر إلى تفاوض تعاقدي/قانوني","provider_assurance":50,"contract_clarity":55,"need":13},
            {"id":"friction","en":"It often slows the deal or creates concern","ar":"غالبًا يؤخر الصفقة أو يسبب قلقًا للعميل","provider_assurance":30,"contract_clarity":40,"need":20},
            {"id":"rare","en":"Clients rarely ask about it","ar":"نادرًا ما يسأل العملاء عنه","provider_assurance":45,"contract_clarity":45,"need":12},
        ]
    },
    {
        "id":"proof", "chapter":4, "paths":["client","provider","hybrid","advisor"],
        "en":"Leadership asks for proof of continuity readiness tomorrow morning. How quickly can you produce it?",
        "ar":"طلبت الإدارة صباح الغد إثبات جاهزية الاستمرارية. كم تحتاجون لإظهاره؟",
        "context_en":"Readiness is not only having controls. It is being able to prove them when it matters.",
        "context_ar":"الجاهزية ليست وجود الضوابط فقط؛ بل قدرتكم على إثباتها وقت الحاجة.",
        "options":[
            {"id":"immediate","en":"Immediately — evidence is organized and current","ar":"فورًا — الأدلة منظمة ومحدثة","governance":100,"need":1},
            {"id":"day","en":"Within a day","ar":"خلال يوم","governance":80,"need":5},
            {"id":"week","en":"Within a week","ar":"خلال أسبوع","governance":60,"need":10},
            {"id":"effort","en":"It would require significant effort","ar":"سيتطلب جهدًا كبيرًا","governance":35,"need":17},
            {"id":"unknown","en":"I do not know where to start","ar":"لا أعرف من أين أبدأ","governance":20,"need":21},
        ]
    },
]


## 5) محرك الـScoring والـPersona
السكور الظاهر للزائر منفصل عن الـCommercial Opportunity المخفي.

In [ ]:
CLIENT_WEIGHTS = {
    "asset_control":0.25, "continuity":0.23, "exit_readiness":0.18,
    "contract_clarity":0.14, "governance":0.10, "business_criticality_inverse":0.10
}
PROVIDER_WEIGHTS = {
    "provider_assurance":0.30, "asset_control":0.20, "continuity":0.20,
    "contract_clarity":0.15, "governance":0.15
}

PERSONAS = {
    "resilient": {
        "en":"The Resilient Guardian", "ar":"الحارس المرن",
        "desc_en":"You appear to have strong control and continuity foundations. The opportunity is to independently validate and sustain that resilience.",
        "desc_ar":"تظهر لديكم أسس قوية للسيطرة والاستمرارية. الفرصة الآن هي التحقق المستقل من هذه المرونة والمحافظة عليها.",
        "service_en":"Resilience Validation & Independent Assurance",
        "service_ar":"التحقق من المرونة والضمان المستقل"
    },
    "managed": {
        "en":"The Managed Dependency", "ar":"الاعتماد المُدار",
        "desc_en":"Most pieces are in place, but a few dependencies could become painful during disruption or transition.",
        "desc_ar":"معظم العناصر موجودة، لكن بعض نقاط الاعتماد قد تتحول إلى ألم فعلي عند التعطل أو الانتقال.",
        "service_en":"Dependency & Continuity Review",
        "service_ar":"مراجعة الاعتماد واستمرارية التقنية"
    },
    "exposed": {
        "en":"The Exposed Operator", "ar":"المشغّل المعرّض",
        "desc_en":"Your operation can run normally today, yet a disruption could expose gaps in access, handover, or contractual continuity.",
        "desc_ar":"التشغيل قد يبدو طبيعيًا اليوم، لكن أي تعطل قد يكشف فجوات في الوصول أو التسليم أو استمرارية العلاقة التعاقدية.",
        "service_en":"Technology Continuity Assessment",
        "service_ar":"تقييم استمرارية التقنية"
    },
    "critical": {
        "en":"The Critical Dependency", "ar":"الاعتماد الحرج",
        "desc_en":"Several signals suggest that continuity still depends heavily on an external relationship. That dependency deserves priority attention before it becomes an incident.",
        "desc_ar":"تشير عدة إشارات إلى أن الاستمرارية ما زالت تعتمد بدرجة كبيرة على طرف خارجي. هذه الفجوة تستحق المعالجة قبل أن تتحول إلى حادث فعلي.",
        "service_en":"Priority Continuity, Escrow & Contract Risk Assessment",
        "service_ar":"تقييم أولوية للاستمرارية وحفظ الأصول ومخاطر العقود"
    },
    "provider_ready": {
        "en":"The Trusted Provider", "ar":"المزوّد الموثوق",
        "desc_en":"You can demonstrate strong continuity and assurance. Independent validation can turn that readiness into a commercial trust signal.",
        "desc_ar":"لديكم قدرة قوية على إثبات الاستمرارية. ويمكن للتحقق المستقل تحويل هذه الجاهزية إلى عنصر ثقة تجاري أمام العملاء.",
        "service_en":"Independent Assurance & Trust Validation",
        "service_ar":"الضمان المستقل والتحقق من الثقة"
    },
    "provider_growth": {
        "en":"The Trust Builder", "ar":"باني الثقة",
        "desc_en":"Your solution may be strong, but continuity assurance could still become friction in enterprise sales. Closing that gap can strengthen trust and shorten negotiations.",
        "desc_ar":"قد يكون الحل قويًا، لكن ضمانات الاستمرارية قد تصبح نقطة احتكاك في مبيعات المؤسسات. إغلاق هذه الفجوة يعزز الثقة ويقلل التفاوض.",
        "service_en":"Provider Assurance & Escrow Readiness",
        "service_ar":"جاهزية ضمان المزوّد وحفظ الأصول"
    },
    "explorer": {
        "en":"The Explorer", "ar":"المستكشف",
        "desc_en":"You are exploring the topic rather than representing an active organizational need today.",
        "desc_ar":"أنت تستكشف الموضوع أكثر من كونك تمثل احتياجًا مؤسسيًا فعليًا حاليًا.",
        "service_en":"Explore Technology Continuity",
        "service_ar":"استكشف مفهوم استمرارية التقنية"
    }
}

def _opt(qid, oid):
    q = next(q for q in QUESTIONS if q['id']==qid)
    return next(o for o in q['options'] if o['id']==oid)

def _avg(selected, metric, default=55):
    vals=[o[metric] for o in selected if metric in o]
    return round(sum(vals)/len(vals),1) if vals else default

def calculate_scores(answers):
    selected=[]
    for q in QUESTIONS:
        if q['id'] in answers:
            selected.append(_opt(q['id'], answers[q['id']]))

    route = next((o.get('route') for o in selected if o.get('route')), 'explorer')
    eligibility = min(100, sum(o.get('eligibility',0)+o.get('fit',0)+o.get('influence',0) for o in selected))
    influence = min(100, sum(o.get('influence',0) for o in selected)*4)
    fit = min(100, sum(o.get('fit',0) for o in selected)*5)
    need_vals=[o.get('need') for o in selected if 'need' in o]
    need=min(100, round((sum(need_vals)/max(1,len(need_vals)))*(100/22)))

    dims={
        'asset_control':_avg(selected,'asset_control'),
        'continuity':_avg(selected,'continuity'),
        'exit_readiness':_avg(selected,'exit_readiness'),
        'contract_clarity':_avg(selected,'contract_clarity'),
        'governance':_avg(selected,'governance'),
        'business_criticality':_avg(selected,'business_criticality',50),
        'provider_assurance':_avg(selected,'provider_assurance',55),
    }

    if route=='provider':
        readiness=sum(dims[k]*w for k,w in PROVIDER_WEIGHTS.items())
    else:
        readiness=(dims['asset_control']*.25+dims['continuity']*.23+dims['exit_readiness']*.18+
                   dims['contract_clarity']*.14+dims['governance']*.10+(100-dims['business_criticality'])*.10)
    readiness=max(0,min(100,round(readiness)))

    commercial=round(.36*need+.20*eligibility+.16*influence+.13*fit+.15*(100-readiness))
    commercial=max(0,min(100,commercial))

    if route=='explorer' or eligibility<20:
        opportunity='General Visitor'
    elif route in ['provider','advisor'] and commercial<65:
        opportunity='Ecosystem Opportunity'
    elif commercial>=75:
        opportunity='Priority Opportunity'
    elif commercial>=55:
        opportunity='Qualified Opportunity'
    else:
        opportunity='Nurture'

    if route=='explorer': persona='explorer'
    elif route=='provider': persona='provider_ready' if readiness>=75 else 'provider_growth'
    elif readiness>=80: persona='resilient'
    elif readiness>=65: persona='managed'
    elif readiness>=45: persona='exposed'
    else: persona='critical'

    candidates=['provider_assurance','asset_control','continuity','contract_clarity','governance'] if route=='provider' else ['asset_control','continuity','exit_readiness','contract_clarity','governance']
    priority=min(candidates,key=lambda x:dims[x])

    return {
        'route':route,'readiness':readiness,'need':need,'eligibility':eligibility,
        'influence':influence,'fit':fit,'commercial':commercial,'opportunity':opportunity,
        'persona_key':persona,'priority_key':priority,'dimensions':dims
    }


## 6) تجربة سريعة على Persona
عدّلي الإجابات وشاهدي كيف تتغير النتيجة.

In [ ]:
demo_answers = {
    'representation':'client',
    'influence':'approve',
    'org_scale':'enterprise',
    'criticality':'minutes',
    'provider_disruption':'wait',
    'handover':'obtain',
    'exit':'provider',
    'proof':'effort'
}

result = calculate_scores(demo_answers)
result

In [ ]:
lang='ar'
p = PERSONAS[result['persona_key']]
print('Persona:', p[lang])
print('Readiness:', result['readiness'])
print('Service Need %:', result['need'])
print('Commercial Opportunity:', result['commercial'])
print('Lead Class:', result['opportunity'])
print('Recommendation:', p['service_ar'] if lang=='ar' else p['service_en'])

## 7) رسم ملف الجاهزية للزائر

In [ ]:
labels = ['Asset Control','Continuity','Exit Readiness','Contract Clarity','Governance']
keys = ['asset_control','continuity','exit_readiness','contract_clarity','governance']
vals = [result['dimensions'][k] for k in keys]
fig = go.Figure(go.Scatterpolar(r=vals+[vals[0]], theta=labels+[labels[0]], fill='toself'))
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,100])), showlegend=False, title='Technology Continuity Profile')
fig.show()

## 8) تخزين النتائج في Google Sheets
### خيار Service Account
1. أنشئي Google Sheet باسم `Technology Continuity Leads`.  
2. أنشئي Service Account من Google Cloud.  
3. شاركي الـSheet مع بريد الـService Account.  
4. ارفعي ملف JSON في Colab عند التشغيل.


In [ ]:
import gspread
from google.oauth2.service_account import Credentials

def connect_google_sheet(service_account_json_path, sheet_name='Technology Continuity Leads'):
    scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive']
    creds=Credentials.from_service_account_file(service_account_json_path, scopes=scopes)
    gc=gspread.authorize(creds)
    return gc.open(sheet_name)

def flatten_record(scores, answers, contact=None):
    contact=contact or {}
    d=scores['dimensions']
    return {
        'timestamp':datetime.now(timezone.utc).isoformat(),
        'session_id':str(uuid.uuid4()),
        'route':scores['route'],
        'persona':scores['persona_key'],
        'readiness_score':scores['readiness'],
        'service_need_pct':scores['need'],
        'eligibility_score':scores['eligibility'],
        'decision_influence_score':scores['influence'],
        'organization_fit_score':scores['fit'],
        'commercial_opportunity_score':scores['commercial'],
        'opportunity_class':scores['opportunity'],
        'priority_area':scores['priority_key'],
        'asset_control':d['asset_control'],
        'continuity':d['continuity'],
        'exit_readiness':d['exit_readiness'],
        'contract_clarity':d['contract_clarity'],
        'governance':d['governance'],
        'business_criticality':d['business_criticality'],
        'provider_assurance':d['provider_assurance'],
        'name':contact.get('name',''),
        'organization':contact.get('organization',''),
        'email':contact.get('email',''),
        'phone':contact.get('phone',''),
        'answers_json':json.dumps(answers, ensure_ascii=False)
    }

def append_to_sheet(sh, record):
    try:
        ws=sh.worksheet('Responses')
    except Exception:
        ws=sh.add_worksheet(title='Responses', rows=3000, cols=40)
    rows=ws.get_all_values()
    headers=list(record.keys())
    if not rows:
        ws.append_row(headers)
    else:
        headers=rows[0]
        for h in record:
            if h not in headers:
                headers.append(h)
        ws.update('1:1',[headers])
    ws.append_row([record.get(h,'') for h in headers], value_input_option='USER_ENTERED')


## 9) Dashboard من بيانات Google Sheets
هذه الدالة تنتج مؤشرات المؤتمر، توزيع الـPersonas، نسبة الاحتياج، وتصنيف الفرص.

In [ ]:
def load_responses(sh):
    ws=sh.worksheet('Responses')
    return pd.DataFrame(ws.get_all_records())

def conference_dashboard(df):
    if df.empty:
        print('No data yet')
        return

    numeric=['readiness_score','service_need_pct','commercial_opportunity_score','eligibility_score']
    for c in numeric:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c], errors='coerce')

    total=len(df)
    qualified=(df['opportunity_class'].isin(['Priority Opportunity','Qualified Opportunity'])).sum()
    priority=(df['opportunity_class']=='Priority Opportunity').sum()
    avg_need=df['service_need_pct'].mean()
    print(f'Total Visitors: {total:,}')
    print(f'Qualified Opportunities: {qualified:,}')
    print(f'Priority Opportunities: {priority:,}')
    print(f'Average Service Need: {avg_need:.1f}%')

    fig1=px.bar(df['persona'].value_counts().reset_index(), x='persona', y='count', title='Persona Distribution')
    fig1.show()

    need_by_persona=df.groupby('persona',as_index=False)['service_need_pct'].mean().sort_values('service_need_pct',ascending=False)
    fig2=px.bar(need_by_persona,x='persona',y='service_need_pct',title='Average Service Need by Persona')
    fig2.show()

    fig3=px.pie(df,names='opportunity_class',title='Opportunity Mix')
    fig3.show()

    fig4=px.scatter(df,x='readiness_score',y='commercial_opportunity_score',color='persona',
                    hover_data=['organization'] if 'organization' in df.columns else None,
                    title='Readiness vs Commercial Opportunity')
    fig4.show()


## 10) منطق الـStorytelling داخل الواجهة
في Streamlit لا نعرض السؤال مباشرة. لكل شاشة نعرض 4 طبقات:

1. **Chapter** — أين وصل في القصة.  
2. **Scene** — موقف قصير.  
3. **Choice** — الاختيارات.  
4. **Micro-reveal** — جملة قصيرة بعد الاختيار مثل: *“Control becomes visible when the provider disappears.”*

هذه الخلية تعطي نصوص الـmicro-reveal المقترحة.

In [ ]:
MICRO_REVEALS = {
    'provider_disruption': {
        'en':'When the provider disappears, control becomes visible.',
        'ar':'عندما يغيب المورد، تظهر حقيقة من يملك السيطرة.'
    },
    'handover': {
        'en':'Ownership is easiest to test at the moment of handover.',
        'ar':'أسهل وقت لاختبار ملكية الأصول هو لحظة التسليم.'
    },
    'exit': {
        'en':'The exit clause matters most when the relationship is no longer comfortable.',
        'ar':'بند الخروج يصبح مهمًا فعلًا عندما لا تعود العلاقة مريحة.'
    },
    'assurance': {
        'en':'Trust grows when continuity can be proven independently.',
        'ar':'الثقة تكبر عندما يمكن إثبات الاستمرارية بشكل مستقل.'
    },
    'proof': {
        'en':'If readiness cannot be evidenced quickly, it may not be operationally ready.',
        'ar':'إذا تعذر إثبات الجاهزية بسرعة، فقد لا تكون جاهزية تشغيلية فعلية.'
    }
}


## 11) نص النتيجة النهائية حسب الـPersona
النتيجة لا تقول للزائر "أنت عندك مشكلة"؛ بل تجعله يكتشف الفجوة ثم تقدم الخدمة كحل منطقي.

In [ ]:
def visitor_result_copy(scores, lang='en'):
    p=PERSONAS[scores['persona_key']]
    priority_labels={
        'asset_control':('Technology Asset Control','السيطرة على الأصول التقنية'),
        'continuity':('Continuity Recovery','التعافي واستمرارية التشغيل'),
        'exit_readiness':('Exit & Transition Readiness','جاهزية الخروج والانتقال'),
        'contract_clarity':('Contract Continuity Clarity','وضوح الاستمرارية في العقود'),
        'governance':('Evidence & Governance','الحوكمة والأدلة'),
        'provider_assurance':('Client Assurance','ضمان العملاء')
    }
    if lang=='ar':
        return {
            'score_title':'درجة جاهزية استمرارية التقنية',
            'persona':p['ar'],
            'story':p['desc_ar'],
            'priority':priority_labels[scores['priority_key']][1],
            'recommendation':p['service_ar'],
            'cta':'اكتشف كيف نغلق هذه الفجوة قبل أن تتحول إلى أزمة'
        }
    return {
        'score_title':'Technology Continuity Readiness',
        'persona':p['en'],
        'story':p['desc_en'],
        'priority':priority_labels[scores['priority_key']][0],
        'recommendation':p['service_en'],
        'cta':'See how to close this gap before it becomes an incident'
    }

visitor_result_copy(result,'ar')

## 12) توليد ملف Streamlit الكامل من النسخة المرفقة
النسخة النهائية للواجهة موجودة كملف Python مستقل مع المشروع. في Colab يمكنك نسخها أو تعديلها ثم نشرها على Streamlit Community Cloud.

**المقترح البصري:** خلفية سوداء أو بيضاء، بطاقات داكنة/فاتحة، Accent بنفسجي + تركوازي، سؤال واحد في كل شاشة، progress خفيف، ونتيجة نهائية كبيرة في المنتصف.